In [0]:
!pip install statsmodels seaborn

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pyspark.sql.functions as F
import pyspark.sql.functions as f

import os
import warnings 
pd.options.display.float_format = '{:.3f}'.format
warnings.filterwarnings("ignore")


In [0]:
%run ../src/config

In [0]:
joined_data_path = "/Volumes/workspace/m5/processed/joined_data"

In [0]:
data = pd.read_parquet(joined_data_path)
data = data.sort_values(by=['id','yearweek'])
data.head()

In [0]:
data.groupby(['state_id']).agg(unique_ids=('id','nunique')).reset_index()

## EDA on California m5 forecasting data

In [0]:
data_ca = data[data['state_id']=="CA"]

In [0]:
print("Rows:", len(data_ca))
print("Series:", data_ca["id"].nunique())
print("Items:", data_ca["item_id"].nunique())
print("Stores:", data_ca["store_id"].nunique())
print("Departments:", data_ca["dept_id"].nunique())
print("Categories:", data_ca["cat_id"].nunique())
print("States:", data_ca["state_id"].nunique())

print(
    data_ca.groupby("id")["yearweek"]
    .nunique()
    .describe()
)

In [0]:
data_ca.duplicated(["id", "yearweek"]).sum()

In [0]:
data_ca.isna().sum()

In [0]:
(32988/1993737)*100

### California Data Overview:
* **12,196 unique time series** (product-store combinations)
* **3,049 unique products** across **4 stores** and **3 categories**
* **178 weeks** of historical data per series (consistent coverage)
* **No duplicate** (id, yearweek) pairs — data integrity is good

**Data Quality:**
* `sell_price` has **1.6% null values** — will need forward-fill or imputation
* All other features are complete

In [0]:
data_ca.groupby(['cat_id'],as_index=False).agg(
    unique_ids=('id', 'nunique'),
    avg_sales=('sales', 'mean'),
    total_sales=('sales', 'sum'),
    sales_std=('sales', 'std'),
    total_weeks=('yearweek', 'nunique'))\
.assign(
    total_sales_m=lambda df: df['total_sales'] / 1e6,
    sales_pct=lambda df: (df['total_sales'] / df['total_sales'].sum()) * 100,
    id_pct=lambda df: (df['unique_ids'] / df['unique_ids'].sum()) * 100
        ).sort_values(by=['sales_pct','id_pct'],ascending=False)

### California Category level Insights:
* FOODS contribute to **65% of sales with 47% of series.**
* HOUSEHOLD contributes to **23% of sales with 34% of series.**
* HOBBIES contributes to **10% of sales with 18% of series.**

## Sales Distribution analysis

In [0]:
print(data_ca["sales"].describe(percentiles=[.5, .75, .9, .95, .99]))

plt.figure(figsize=(10,5))

plt.hist(
    np.log1p(data_ca["sales"]),
    bins=100
)

plt.xlabel("log1p(Sales)")
plt.ylabel("Frequency")
plt.title("Sales Distribution")

plt.show()

In [0]:
zero_share = (data_ca.groupby("id")["sales"].apply(lambda x: (x == 0).mean()))

print(zero_share.describe(percentiles=[.5, .75, .9, .95, .99]))

plt.figure(figsize=(10,5))

plt.hist(
    zero_share,
    bins=50
)

plt.xlabel("Zero Sales Share")
plt.ylabel("Number of Series")
plt.title("Demand Intermittency")

plt.show()

### Sales Distribution Insights

**Highly Right-Skewed Distribution:**
* **Median:** 4 units/week
* **Mean:** 10 units/week (pulled up by high-volume items)
* **99th percentile:** 102 units (some very high-volume series)

**Intermittency Analysis:**
* **Average zero-share:** 23.6% (many weeks have zero sales)
* Distribution ranges from 0% to 97% zero weeks
* ~1,300 series have very low intermittency (<5% zeros)
* Long tail of highly intermittent series

**Implication:** Log transformation useful for modeling; intermittent demand methods needed for sparse series.

## segment analysis

In [0]:
data_ca_segments = compute_segments(data_ca)
data_ca_segments.head().display()

In [0]:
data_segmented = data_ca.merge(data_ca_segments[['id','segment']], how='left', on='id')
data_segmented.head().display()

In [0]:
segment_summary = (
    data_segmented
    .groupby("segment")
    .agg(
        series_count=("id", "nunique"),
        total_sales=("sales", "sum"),
        avg_sales=("sales", "mean")
    ).reset_index()
)

segment_summary["series_pct"] = (
    segment_summary["series_count"]
    / segment_summary["series_count"].sum()
    * 100
)

segment_summary["sales_pct"] = (
    segment_summary["total_sales"]
    / segment_summary["total_sales"].sum()
    * 100
)

segment_summary.sort_values(
    "total_sales",
    ascending=False
)

In [0]:
segment_category_summary = (
    data_segmented
    .groupby(['cat_id',"segment"])
    .agg(
        series_count=("id", "nunique"),
        total_sales=("sales", "sum"),
        avg_sales=("sales", "mean")
    ).reset_index()
)

segment_category_summary["series_pct"] = (
    segment_category_summary["series_count"]
    / segment_category_summary["series_count"].sum()
    * 100
)

segment_category_summary["sales_pct"] = (
    segment_category_summary["total_sales"]
    / segment_category_summary["total_sales"].sum()
    * 100
)

segment_category_summary.sort_values(
    "total_sales",
    ascending=False
)

In [0]:
data_segmented.groupby("segment")["sell_price"].describe().sort_values(by=['mean'],ascending=False)

### Segmentation level Insights

**Segment Distribution & Sales Contribution:**

| Segment | Series % | Sales % | Avg Sales | Modeling Priority |
|---------|----------|---------|-----------|-------------------|
| **Smooth** | 35.13% | **62.77%** | 17.56 | High — drives majority of revenue |
| **Erratic** | 23.88% | 23.08% | 9.59 | Medium — volatile but significant |
| **Lumpy** | 40.00% | 13.93% | 3.51 | Medium — largest group by count |
| **New** | 0.93% | 0.14% | 6.72 | Low — insufficient history |
| **Intermittent** | 0.02% | 0.08% | 31.46 | Low — rare events |
| **Dead** | 0.04% | 0.00% | 0.13 | Exclude — no demand |

**Category × Segment Breakdown:**
* **FOODS + Smooth:** 39.6% of total sales (top performer)
* **FOODS + Erratic:** 17.1% of sales
* **HOUSEHOLD + Smooth:** 16.1% of sales

**Price Variation by Segment:**
* Lumpy items have slightly higher prices (mean 4.97)
* Smooth items average 4.14
* Intermitent items have lowest avg prices (3.22)

**Key Takeaway:** Different segments need different modeling approaches — smooth series can use traditional time series methods, while lumpy/erratic need specialized intermittent demand forecasting.

## Price Analysis

In [0]:
data_ca["sell_price"].describe()

In [0]:
price_stats = (
    data_ca.groupby("id",as_index=True)
    .agg(
        avg_price=("sell_price", "mean"),
        min_price=("sell_price", "min"),
        max_price=("sell_price", "max"),
        price_std=("sell_price", "std")
    ).reset_index()
)

price_stats["price_cv"] = (
    price_stats["price_std"] /
    price_stats["avg_price"]
)

price_stats.describe()

In [0]:
data_ca[
    ["sales", "sell_price"]
].corr()

### Price Analysis Insights

**Price Distribution:**
* **Mean price:** 4.45, **Median:** 3.48 (right-skewed)
* Range:0.01 to30.98 (wide variety)

**Price Variability (within products over time):**
* **Mean CV:** 0.2% (prices change moderately)
* most of the product*cutomers have stable prices (CV < 10%)

**Price-Sales Correlation:**
* **Pearson correlation: -0.164** (weak negative)
* Higher prices → slightly lower sales
* Suggests **price elasticity exists** but other factors (seasonality, events) dominate

**Implication:** Price features (absolute, changes, promotions) should be included but won't be the primary driver.

## Event analysis

In [0]:
data_ca.groupby('event_flag').agg(sales=('sales','sum')).reset_index().assign(sales_pct = lambda df:df['sales']/df['sales'].sum() *100)

In [0]:
event_cols = [c for c in data_ca.columns if c.startswith("f_event_")]

lift_results = []
baseline_sales = data_ca[data_ca[event_cols].sum(axis=1) == 0]["sales"].mean()
print(f"Baseline sales {baseline_sales}")

for c in event_cols:
  event_sales = data_ca[data_ca[c] == 1]["sales"].mean()
  pct_lift = ((event_sales - baseline_sales) / baseline_sales) * 100
  lift_results.append(
      {"event": c, "weeks": data_ca[c].sum(), "avg_sales_lift%": pct_lift,"avg_sales_change_abs%":np.abs(pct_lift)}
  )

event_impact_df = pd.DataFrame(lift_results).sort_values(
    "avg_sales_change_abs%", ascending=False
)

event_impact_df.head(5).display()

### Event Impact Analysis

**Top Events by Sales Lift:**

| Event | Lift % | Interpretation |
|-------|--------|----------------|
| Christmas | -22% | Strong impact -  sales reduced  |
| New year | -22% | Strong impact- sales reduced |
| thanksgiving| -14% | strong impact- sales reduced |
| laborday | +12% | strong impact - sales increased | 
| independence day| +12% | strong impact - sales increased|

**Event Coverage:**
* Most events occur in 30-45K week observations
* Events are relatively well-balanced in frequency

**Key Finding:** Major holidays (Christmas, NewYear, Thanksgiving, LaborDay) have substantial lift. Event flags are **valuable features** for modeling, especially for food categories.

In [0]:
data_ca['year'] = data_ca['yearweek'].astype(str).str[:4].astype(int)
data_ca['week'] = data_ca['yearweek'].astype(str).str[4:].astype(int)

weekly_trend = (
    data_ca.groupby(['cat_id', 'year', 'week'])['sales']
    .sum()
    .reset_index()
)

weekly_trend = weekly_trend[weekly_trend['week'] != 53]
weekly_trend.display()

In [0]:
categories = weekly_trend['cat_id'].unique()

sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(
    nrows=len(categories), ncols=1, figsize=(10, 4 * len(categories)), sharex=True
)

for i, cat in enumerate(categories):
  ax = axes[i]
  cat_data = weekly_trend[weekly_trend['cat_id'] == cat]

  sns.lineplot(
      data=cat_data,
      x='week',
      y='sales',
      hue='year',
      palette='Set1',
      linewidth=2,
      alpha=0.85,
      ax=ax,
  )

  ax.set_title(
      f'Year-over-Year Weekly Seasonality — Category: {cat}',
      fontsize=12,
      fontweight='bold',
  )
  ax.set_ylabel('Total Weekly Sales')

  if i == len(categories) - 1:
    ax.set_xlabel('Week of the Year (1-52)', fontsize=11)
  else:
    ax.set_xlabel('')

  ax.legend(title='Year', loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

### Seasonality Patterns

**Overall Weekly Sales Trend:**
* Clear **upward trend** from 2013 to 2016
* Some **volatility spikes** (likely holiday weeks)
* No obvious structural breaks or regime changes

**Implication:** Seasonality is strong and should be captured via:
* Week-of-year features
* Fourier terms for smooth seasonal cycles
* Lag-52 features (year-over-year comparison)

## Feature Engineering Strategy

Based on the EDA, here are the recommended features to engineer:

### 1. **Lag Features** (Critical for Time Series)
* `sales_lag_1`, `sales_lag_4`, `sales_lag_8` (recent history)
* `sales_lag_52` (year-over-year comparison)
* `sales_lag_7` (if daily data, week-over-week)

### 2. **Rolling Window Statistics**
* **Rolling mean:** 4, 8, 13, 26 weeks (capture trend)
* **Rolling std:** 4, 8 weeks (capture volatility)
* **Rolling min/max:** 4, 8 weeks (range of demand)
* **Rolling zero-count:** 4, 8 weeks (intermittency tracking)

### 3. **Price-Derived Features**
* `price_change_pct` (week-over-week % change)
* `price_vs_category_median` (relative pricing)
* `is_discount` (binary: price < rolling avg)
* `price_lag_1` (previous week's price)

### 4. **Calendar Features**
* ISO week number (1-52)
* Month (1-12)
* Quarter (Q1-Q4)
* Year (for trend)
* Days-since-first-sale (product lifecycle)

### 5. **Expanding Features**
* Expanding mean sales per `id` (lifetime average)
* Expanding max sales (normalization reference)

### 6. **Target Encoding**
* Mean sales by `item_id`, `store_id`, `dept_id`, `cat_id`
* Use proper CV to avoid leakage

### 7. **Intermittency Metrics**
* **ADI** (Average Demand Interval): avg weeks between non-zero sales
* **CV²** (Coefficient of Variation squared): variability of non-zero demand
* Use to classify series: smooth (<1.32 ADI, <0.49 CV²), erratic, lumpy, intermittent

### 8. **Segment-Based Features**
* One-hot encode `segment` (smooth, erratic, lumpy)
* Segment-specific aggregate stats

### 9. **Event Features**
* All `f_event_*` flags (already present)
* `event_flag` (any event binary)
* Event interaction with category

### 10. **SNAP Benefits** (if available)
* `snap_CA_days` (already present)
* Rolling sum of SNAP days (4-week window)



## Modeling Strategy

### **Phase 1: Baseline Models**
Establish performance benchmarks:

1. **Naive Forecast:** Last week's sales
2. **Seasonal Naive:** Sales from same week last year (lag-52)
3. **Simple Moving Average:** 4-week and 8-week MA
4. **Exponential Smoothing (ETS):** For smooth series

**Evaluation Metric:** WRMSSE (Weighted Root Mean Squared Scaled Error) — the M5 competition metric

---

### **Phase 2: Segment-Specific Models**
Different segments need different approaches:

| Segment | Recommended Models | Rationale |
|---------|-------------------|----------|
| **Smooth** (63% of sales) | ARIMA, ETS, Prophet, LightGBM | Traditional TS methods work well |
| **Erratic** (23% of sales) | LightGBM, XGBoost with rich features | ML handles volatility better |
| **Lumpy** (14% of sales) | Croston, TSB, ADIDA, LightGBM | Intermittent demand methods |
| **New** (0.1% of sales) | Category-level model or holdout | Insufficient history |
| **Dead** (0% of sales) | Zero forecast or exclude | No demand |

---

### **Phase 3: Machine Learning Models**
For the bulk of series (smooth + erratic):

**Model 1: LightGBM (Primary)**
* Use all engineered features
* Group by `id` for proper time series split
* Hyperparameter tuning: `num_leaves`, `learning_rate`, `min_data_in_leaf`
* Train separate models by category or segment

**Model 2: XGBoost (Secondary)**
* Similar feature set
* Compare performance with LightGBM

**Model 3: Neural Network (Optional)**
* LSTM or Temporal Fusion Transformer for smooth series
* Requires significant tuning

---

### **Phase 4: Ensemble & Hierarchy**

1. **Ensemble Forecasts:**
   * Weighted average of LightGBM + ARIMA + Croston (by segment)
   * Optimize weights on validation set

2. **Hierarchical Reconciliation:**
   * Bottom-up: Sum store-level forecasts to category/state level
   * Top-down: Forecast at category level, disaggregate by historical proportions
   * Optimal reconciliation: MinT (Minimum Trace)

---

### **Phase 5: Train/Test Strategy**

**Time-Based Split (Critical for Time Series):**
* **Training:** Weeks 1-160 (90%)
* **Validation:** Weeks 161-170 (5-6%)
* **Test (Holdout):** Weeks 171-178 (last 8 weeks)

**Cross-Validation:**
* Use **expanding window** or **rolling window** CV
* Respect temporal order — never train on future data
* Example: Train on weeks 1-130, validate on 131-140; then 1-140, validate on 141-150

**Evaluation:**
* Primary: **WRMSSE** (weighted by sales $ and scaled by series volatility)
* Secondary: RMSE, MAE, MAPE (by segment and category)
* Check for **bias:** under-forecasting vs over-forecasting

---

### **Phase 6: Model Monitoring & Productionization**

1. **Retrain Cadence:** Weekly or bi-weekly with latest data
2. **Drift Detection:** Monitor actual vs forecast, flag large deviations
3. **Segment Stability:** Re-run segmentation periodically
4. **Feature Importance:** Track which features drive predictions
5. **Outlier Handling:** Cap extreme predictions to historical max × 1.5


## Next Steps

1. ✅ **EDA Complete** — this notebook
2. ⬜ **Feature Engineering** — create lag, rolling, price, calendar features
3. ⬜ **Baseline Models** — naive, seasonal naive, MA
4. ⬜ **Segment Models** — ARIMA/ETS for smooth, Croston for lumpy
5. ⬜ **ML Models** — LightGBM with full feature set
6. ⬜ **Ensemble** — weighted combination of models
7. ⬜ **Evaluation** — WRMSSE on holdout set
8. ⬜ **Production** — deploy best model(s)